# 1.1 · VaR con Monte Carlo

**Objetivo:** estimar VaR y CVaR de una cartera simulando retornos multivariantes.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parents[1]))  # repo root, so `common` is importable
%load_ext autoreload
%autoreload 2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from common import *
set_plot_style()

## 1. Datos
Descarga precios de una cartera y calcula retornos logarítmicos diarios.

In [ ]:
tickers = ["AAPL", "MSFT", "JPM", "XOM", "GLD"]
weights = np.array([0.25, 0.25, 0.2, 0.15, 0.15])
prices = download_prices(tickers, start="2018-01-01")
rets = to_returns(prices)
rets.tail()

## 2. Ejercicio — VaR paramétrico (normal)
Calcula el VaR al 95% a 1 día con $\mu_p$ y $\sigma_p$ de la cartera. Devuelve una pérdida positiva.

In [ ]:
alpha = 0.95
port = rets @ weights
# TODO: var_param = ...


In [ ]:
from scipy.stats import norm
var_param = -(port.mean() + norm.ppf(1 - alpha) * port.std())
print(f"VaR paramétrico {alpha:.0%} 1d: {var_param:.2%}")

## 3. Ejercicio — Simulación Monte Carlo
Simula 100.000 retornos usando la media y covarianza de `rets` (descomposición de Cholesky) y calcula VaR y CVaR.

In [ ]:
n_sims = 100_000
rng = np.random.default_rng(42)
# TODO: sim_port = ...


In [ ]:
mu, cov = rets.mean().values, rets.cov().values
L = np.linalg.cholesky(cov)
z = rng.standard_normal((n_sims, len(tickers)))
sim = mu + z @ L.T
sim_port = sim @ weights
var_mc = -np.quantile(sim_port, 1 - alpha)
cvar_mc = -sim_port[sim_port <= -var_mc].mean()
print(f"VaR MC {alpha:.0%}: {var_mc:.2%} | CVaR: {cvar_mc:.2%}")

## 4. Visualización

In [ ]:
fig, ax = plt.subplots()
ax.hist(sim_port, bins=100, alpha=0.7)
ax.axvline(-var_mc, color="r", ls="--", label=f"VaR {alpha:.0%}")
ax.axvline(-cvar_mc, color="k", ls="--", label="CVaR")
ax.set_title("Distribución simulada del retorno diario"); ax.legend(); plt.show()